In [1]:
from pathlib import Path
import pandas as pd

print("02.19 STEP 1 - DUPLICATE DATA INSPECTION")
print("=" * 65)

BASE_DIR = Path.cwd().parent

INPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "missing_value_adjusted"
    / "sales_adjusted"
)

sales_files = sorted(INPUT_DIR.glob("sales_*.csv"))

print("\nSales Data")
print("-" * 65)
print("Sales files found:", len(sales_files))

total_rows = 0
total_duplicate_sales_line_ids = 0

for file in sales_files:
    df = pd.read_csv(file)

    total_rows += len(df)

    duplicate_ids = df["sales_line_id"].duplicated().sum()
    total_duplicate_sales_line_ids += duplicate_ids

print("Total sales rows:", total_rows)
print("Existing duplicate sales_line_id rows:", total_duplicate_sales_line_ids)

print("\nExpected")
print("-" * 65)
print("Expected files: 32")
print("Expected rows: 2,118,200")

if (
    len(sales_files) == 32
    and total_rows == 2_118_200
    and total_duplicate_sales_line_ids == 0
):
    print("\n02.19 Step 1 inspection: PASSED")
else:
    print("\n02.19 Step 1 inspection: FAILED")

02.19 STEP 1 - DUPLICATE DATA INSPECTION

Sales Data
-----------------------------------------------------------------
Sales files found: 32
Total sales rows: 2118200
Existing duplicate sales_line_id rows: 0

Expected
-----------------------------------------------------------------
Expected files: 32
Expected rows: 2,118,200

02.19 Step 1 inspection: PASSED


In [2]:
print("02.19 STEP 2 - DEFINE DUPLICATE SCENARIO")
print("=" * 65)

TOTAL_SALES_ROWS = 2_118_200

# Controlled duplicate rate
DUPLICATE_RATE = 0.005   # 0.50%

# Number of rows to duplicate
DUPLICATE_ROWS = int(TOTAL_SALES_ROWS * DUPLICATE_RATE)

print("\nDuplicate Scenario")
print("-" * 65)
print("Total sales rows:", TOTAL_SALES_ROWS)
print("Duplicate rate:", f"{DUPLICATE_RATE * 100:.2f}%")
print("Rows to duplicate:", DUPLICATE_ROWS)

# Validation
expected_duplicate_rows = int(TOTAL_SALES_ROWS * 0.005)

print("\nValidation")
print("-" * 65)
print(
    "Duplicate rate valid:",
    DUPLICATE_RATE == 0.005
)

print(
    "Duplicate row count valid:",
    DUPLICATE_ROWS == expected_duplicate_rows
)

if (
    DUPLICATE_RATE == 0.005
    and DUPLICATE_ROWS == expected_duplicate_rows
):
    print("\n02.19 Step 2 validation: PASSED")
else:
    print("\n02.19 Step 2 validation: FAILED")

02.19 STEP 2 - DEFINE DUPLICATE SCENARIO

Duplicate Scenario
-----------------------------------------------------------------
Total sales rows: 2118200
Duplicate rate: 0.50%
Rows to duplicate: 10591

Validation
-----------------------------------------------------------------
Duplicate rate valid: True
Duplicate row count valid: True

02.19 Step 2 validation: PASSED


In [4]:
from pathlib import Path
import pandas as pd

print("02.19 STEP 3 - CORRECT DUPLICATE INJECTION")
print("=" * 65)

BASE_DIR = Path.cwd().parent

INPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "missing_value_adjusted"
    / "sales_adjusted"
)

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "duplicate_row_adjusted"
    / "sales_adjusted"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_DUPLICATES = 10_591

sales_files = sorted(INPUT_DIR.glob("sales_*.csv"))

# Load all files
dataframes = {}
total_rows = 0

for file in sales_files:
    df = pd.read_csv(file)
    dataframes[file.name] = df
    total_rows += len(df)

print("Input files:", len(sales_files))
print("Original rows:", total_rows)
print("Target duplicate rows:", TARGET_DUPLICATES)

# Calculate base allocation
base_duplicates = {
    name: int(len(df) * TARGET_DUPLICATES / total_rows)
    for name, df in dataframes.items()
}

allocated = sum(base_duplicates.values())
remaining = TARGET_DUPLICATES - allocated

# Distribute remaining rows across files
for name in list(base_duplicates.keys())[:remaining]:
    base_duplicates[name] += 1

# Create adjusted files
total_duplicate_rows = 0
total_adjusted_rows = 0

for file_name, df in dataframes.items():

    duplicate_count = base_duplicates[file_name]

    duplicate_rows = df.sample(
        n=duplicate_count,
        random_state=42
    ).copy()

    adjusted_df = pd.concat(
        [df, duplicate_rows],
        ignore_index=True
    )

    adjusted_df.to_csv(
        OUTPUT_DIR / file_name,
        index=False
    )

    total_duplicate_rows += duplicate_count
    total_adjusted_rows += len(adjusted_df)

print("\nCorrected Injection Summary")
print("-" * 65)
print("Files processed:", len(sales_files))
print("Original rows:", total_rows)
print("Duplicate rows injected:", total_duplicate_rows)
print("Adjusted rows:", total_adjusted_rows)

print("\nOutput folder:")
print(OUTPUT_DIR)

if (
    total_duplicate_rows == TARGET_DUPLICATES
    and total_adjusted_rows == total_rows + TARGET_DUPLICATES
):
    print("\n02.19 Step 3 correction: PASSED")
else:
    print("\n02.19 Step 3 correction: FAILED")

02.19 STEP 3 - CORRECT DUPLICATE INJECTION
Input files: 32
Original rows: 2118200
Target duplicate rows: 10591

Corrected Injection Summary
-----------------------------------------------------------------
Files processed: 32
Original rows: 2118200
Duplicate rows injected: 10591
Adjusted rows: 2128791

Output folder:
c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\duplicate_row_adjusted\sales_adjusted

02.19 Step 3 correction: PASSED


In [6]:
from pathlib import Path
import pandas as pd

print("02.19 STEP 4 - CORRECT DUPLICATE VALIDATION")
print("=" * 65)

BASE_DIR = Path.cwd().parent

INPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "duplicate_row_adjusted"
    / "sales_adjusted"
)

sales_files = sorted(INPUT_DIR.glob("sales_*.csv"))

total_rows = 0
total_duplicate_rows = 0
files_with_duplicates = 0

for file in sales_files:

    df = pd.read_csv(file)

    total_rows += len(df)

    # Count only the additional duplicate copies,
    # not the original occurrence.
    duplicate_rows = df["sales_line_id"].duplicated(
        keep="first"
    ).sum()

    if duplicate_rows > 0:
        files_with_duplicates += 1

    total_duplicate_rows += duplicate_rows

print("\nValidation Results")
print("-" * 65)

print("Sales files:", len(sales_files))
print("Total rows:", total_rows)
print("Files containing duplicates:", files_with_duplicates)
print("Additional duplicate rows identified:", total_duplicate_rows)

expected_total_rows = 2_128_791
expected_duplicate_rows = 10_591

print("\nExpected")
print("-" * 65)

print("Expected total rows:", expected_total_rows)
print("Expected additional duplicate rows:", expected_duplicate_rows)

validation_passed = (
    len(sales_files) == 32
    and total_rows == expected_total_rows
    and total_duplicate_rows == expected_duplicate_rows
)

if validation_passed:
    print("\n02.19 Step 4 validation: PASSED")
else:
    print("\n02.19 Step 4 validation: FAILED")

02.19 STEP 4 - CORRECT DUPLICATE VALIDATION

Validation Results
-----------------------------------------------------------------
Sales files: 32
Total rows: 2128791
Files containing duplicates: 32
Additional duplicate rows identified: 10591

Expected
-----------------------------------------------------------------
Expected total rows: 2128791
Expected additional duplicate rows: 10591

02.19 Step 4 validation: PASSED


In [7]:
from pathlib import Path
import pandas as pd

print("02.19 STEP 5 - FINAL DUPLICATE VALIDATION")
print("=" * 65)

BASE_DIR = Path.cwd().parent

OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "synthetic"
    / "duplicate_row_adjusted"
    / "sales_adjusted"
)

EXPECTED_FILES = 32
EXPECTED_ORIGINAL_ROWS = 2_118_200
EXPECTED_DUPLICATE_ROWS = 10_591
EXPECTED_TOTAL_ROWS = (
    EXPECTED_ORIGINAL_ROWS + EXPECTED_DUPLICATE_ROWS
)

sales_files = sorted(OUTPUT_DIR.glob("sales_*.csv"))

total_rows = 0
total_additional_duplicates = 0
files_with_duplicates = 0

for file in sales_files:

    df = pd.read_csv(file)

    total_rows += len(df)

    duplicate_rows = df.duplicated(
        subset=["sales_line_id"],
        keep="first"
    ).sum()

    total_additional_duplicates += duplicate_rows

    if duplicate_rows > 0:
        files_with_duplicates += 1


print("\nFinal Validation Results")
print("-" * 65)

print("Sales files:", len(sales_files))
print("Total rows:", total_rows)
print("Files containing duplicates:", files_with_duplicates)
print("Additional duplicate rows:", total_additional_duplicates)

print("\nExpected")
print("-" * 65)

print("Expected files:", EXPECTED_FILES)
print("Expected total rows:", EXPECTED_TOTAL_ROWS)
print("Expected additional duplicate rows:", EXPECTED_DUPLICATE_ROWS)


validation_passed = (
    len(sales_files) == EXPECTED_FILES
    and total_rows == EXPECTED_TOTAL_ROWS
    and total_additional_duplicates == EXPECTED_DUPLICATE_ROWS
    and files_with_duplicates == EXPECTED_FILES
)

if validation_passed:
    print("\n02.19 Step 5 validation: PASSED")
else:
    print("\n02.19 Step 5 validation: FAILED")

02.19 STEP 5 - FINAL DUPLICATE VALIDATION

Final Validation Results
-----------------------------------------------------------------
Sales files: 32
Total rows: 2128791
Files containing duplicates: 32
Additional duplicate rows: 10591

Expected
-----------------------------------------------------------------
Expected files: 32
Expected total rows: 2128791
Expected additional duplicate rows: 10591

02.19 Step 5 validation: PASSED
